# REMI+ Bar Structure Exploration

Goal: understand how `pretty_midi` exposes bar/beat structure and what per-bar features
are available for expert descriptions. This feeds into:
1. A REMI+ tokenizer (`BAR`, `POSITION`, `TIME_SIG` tokens)
2. An expert description extractor (per-bar note density, pitch range, velocity, etc.)

Key pretty_midi APIs we'll use:
- `pm.time_signature_changes` — list of `TimeSignature` objects
- `pm.get_downbeats()` — bar start times in seconds
- `pm.get_beats()` — beat times in seconds
- `pm.get_tempo_changes()` — (times, tempos) arrays
- `pm.estimate_tempo()` — single BPM estimate

In [ ]:
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Point this at any piano MIDI file you have locally
# ADL or Lakh both work
SAMPLE_FILE = "../../../data/lakh_clean/Gilbert_and_Sullivan/When_a_merry_maiden_marries.mid"

pm = pretty_midi.PrettyMIDI(SAMPLE_FILE)
print(f"Duration: {pm.get_end_time():.2f}s")

## 1. Time Signatures

`pm.time_signature_changes` is a list of `TimeSignature` objects, each with:
- `.numerator` — beats per bar (e.g. 4 in 4/4)
- `.denominator` — beat unit (e.g. 4 means quarter note, 8 means eighth note)
- `.time` — seconds at which this time signature takes effect

Most files have a single entry at t=0. Some have multiple changes mid-piece.

In [ ]:
print(f"Time signature changes: {len(pm.time_signature_changes)}")
for ts in pm.time_signature_changes:
    print(f"  t={ts.time:.3f}s  {ts.numerator}/{ts.denominator}")

## 2. Bar Boundaries via get_downbeats()

`pm.get_downbeats()` returns a numpy array of bar start times (seconds).
It uses the tempo map and time signature changes internally — no manual math needed.

This is the primary API we'll use to segment notes into bars.

In [ ]:
downbeats = pm.get_downbeats()
print(f"Total bars: {len(downbeats)}")
print(f"First 10 bar start times (s): {downbeats[:10].round(3)}")

# Bar durations
bar_durations = np.diff(downbeats)
print(f"\nBar duration — mean: {bar_durations.mean():.3f}s  std: {bar_durations.std():.3f}s")
print(f"Bar duration — min: {bar_durations.min():.3f}s  max: {bar_durations.max():.3f}s")

## 3. Beat Positions Within a Bar

`pm.get_beats()` returns all beat times. Beats within a bar tell us rhythmic position.

For REMI+, we want to express note position as a `POSITION` token: which sub-division
of the bar the note falls on. The standard approach uses a fixed resolution (e.g. 16
sub-divisions per bar regardless of time signature), which keeps the vocab size fixed.

Alternative: use beat-relative position (0 to numerator-1), which is musically meaningful
but requires knowing the current time signature.

In [ ]:
beats = pm.get_beats()
print(f"Total beats: {len(beats)}")

# Show how beats nest inside the first 3 bars
for bar_idx in range(min(3, len(downbeats) - 1)):
    bar_start = downbeats[bar_idx]
    bar_end   = downbeats[bar_idx + 1]
    beats_in_bar = beats[(beats >= bar_start) & (beats < bar_end)]
    print(f"  Bar {bar_idx}: start={bar_start:.3f}s  beats={len(beats_in_bar)}  "
          f"beat_times={np.round(beats_in_bar - bar_start, 3)}")

## 4. Assigning Notes to Bars

For both REMI+ tokenization and expert descriptions we need to know which bar
each note belongs to. `np.searchsorted` on the downbeats array is efficient.

In [ ]:
# Collect all notes across instruments
all_notes = []
for instr in pm.instruments:
    for note in instr.notes:
        all_notes.append((note.start, note.end, note.pitch, note.velocity))
all_notes.sort()

note_starts = np.array([n[0] for n in all_notes])

# bar_index[i] = which bar note i falls in (0-indexed)
# side='right' means a note exactly on a downbeat belongs to that bar
bar_indices = np.searchsorted(downbeats, note_starts, side='right') - 1
bar_indices = np.clip(bar_indices, 0, len(downbeats) - 1)

print(f"Total notes: {len(all_notes)}")
print(f"Unique bars with notes: {len(np.unique(bar_indices))} / {len(downbeats)} bars")

# Notes per bar distribution
notes_per_bar = np.bincount(bar_indices, minlength=len(downbeats))
print(f"Notes/bar — mean: {notes_per_bar.mean():.1f}  median: {np.median(notes_per_bar):.0f}  "
      f"max: {notes_per_bar.max()}")

## 5. Per-Bar Expert Description Features

These are the features we'll encode as expert description tokens prepended to each bar:

| Feature | What it captures | Token idea |
|---|---|---|
| Note density | How busy the bar is | `DENSITY_<bin>` |
| Pitch mean / range | Melodic register | `PITCH_LOW_<bin>`, `PITCH_HIGH_<bin>` |
| Velocity mean | Loudness / dynamics | `VELOCITY_MEAN_<bin>` |
| Polyphony mean | Chord density | `POLYPHONY_<bin>` |
| Time signature | Rhythmic feel | `TIME_SIG_4_4` etc. |

In [ ]:
def get_time_sig_at(pm, t):
    """Return the active TimeSignature at time t."""
    active = pm.time_signature_changes[0]
    for ts in pm.time_signature_changes:
        if ts.time <= t:
            active = ts
    return active


def compute_bar_features(pm, all_notes, downbeats, bar_indices):
    """Compute per-bar expert description features.
    
    Returns a list of dicts, one per bar.
    """
    n_bars = len(downbeats)
    bar_features = []

    for bar_idx in range(n_bars):
        bar_start = downbeats[bar_idx]
        bar_end   = downbeats[bar_idx + 1] if bar_idx + 1 < n_bars else pm.get_end_time()
        bar_dur   = bar_end - bar_start

        mask = bar_indices == bar_idx
        bar_notes = [all_notes[i] for i in np.where(mask)[0]]

        ts = get_time_sig_at(pm, bar_start)

        if bar_notes:
            pitches    = [n[2] for n in bar_notes]
            velocities = [n[3] for n in bar_notes]

            # Polyphony: count simultaneous notes at each note-on time
            # (approximate: count notes whose start time clusters together)
            starts = np.array([n[0] for n in bar_notes])
            ends   = np.array([n[1] for n in bar_notes])
            poly_counts = []
            for s in starts:
                poly_counts.append(((starts <= s) & (ends > s)).sum())

            features = {
                "bar_idx":        bar_idx,
                "bar_start":      round(bar_start, 4),
                "bar_dur":        round(bar_dur, 4),
                "time_sig":       f"{ts.numerator}/{ts.denominator}",
                "note_count":     len(bar_notes),
                "note_density":   round(len(bar_notes) / bar_dur, 2),  # notes/sec
                "pitch_min":      min(pitches),
                "pitch_max":      max(pitches),
                "pitch_mean":     round(np.mean(pitches), 2),
                "velocity_mean":  round(np.mean(velocities), 2),
                "polyphony_mean": round(np.mean(poly_counts), 2),
            }
        else:
            features = {
                "bar_idx":        bar_idx,
                "bar_start":      round(bar_start, 4),
                "bar_dur":        round(bar_dur, 4),
                "time_sig":       f"{ts.numerator}/{ts.denominator}",
                "note_count":     0,
                "note_density":   0.0,
                "pitch_min":      0,
                "pitch_max":      0,
                "pitch_mean":     0.0,
                "velocity_mean":  0.0,
                "polyphony_mean": 0.0,
            }

        bar_features.append(features)

    return bar_features


bar_features = compute_bar_features(pm, all_notes, downbeats, bar_indices)

print("First 5 bars:")
for f in bar_features[:5]:
    print(f"  bar {f['bar_idx']:3d}  {f['time_sig']}  notes={f['note_count']:3d}  "
          f"density={f['note_density']:5.1f}/s  pitch=[{f['pitch_min']},{f['pitch_max']}]  "
          f"vel={f['velocity_mean']:.0f}  poly={f['polyphony_mean']:.1f}")

## 6. Note Position Within Bar

For REMI+ `POSITION` tokens, we need to quantize each note's offset within its bar
to a fixed grid. Standard choice: 16 positions per bar (16th-note resolution).

This works regardless of time signature — a 3/4 bar and a 4/4 bar both map to
positions 0–15. The model learns the effective rhythmic density from the pattern.

Alternative: use beat-relative position (0 to numerator-1). More interpretable but
requires the model to know the current time sig at each position token.

In [ ]:
POSITION_RESOLUTION = 16  # positions per bar

def get_position_in_bar(note_time, bar_start, bar_end, resolution=POSITION_RESOLUTION):
    """Quantize note onset to position 0..resolution-1 within its bar."""
    bar_dur = bar_end - bar_start
    if bar_dur <= 0:
        return 0
    offset = (note_time - bar_start) / bar_dur
    offset = max(0.0, min(offset, 1.0 - 1e-9))  # clamp to [0, 1)
    return int(offset * resolution)


# Show position distribution for the first bar with notes
bar_with_notes = next(i for i, f in enumerate(bar_features) if f["note_count"] > 0)
b_start = downbeats[bar_with_notes]
b_end   = downbeats[bar_with_notes + 1] if bar_with_notes + 1 < len(downbeats) else pm.get_end_time()

bar_notes_sample = [n for n in all_notes if b_start <= n[0] < b_end]
positions = [get_position_in_bar(n[0], b_start, b_end) for n in bar_notes_sample]

print(f"Bar {bar_with_notes} ({bar_features[bar_with_notes]['time_sig']}):")
for note, pos in zip(bar_notes_sample[:10], positions[:10]):
    print(f"  pitch={note[2]:3d}  offset={note[0]-b_start:.3f}s  position={pos:2d}/{POSITION_RESOLUTION}")

# Position histogram across the whole file
all_positions = []
for bar_idx in range(len(downbeats)):
    b_s = downbeats[bar_idx]
    b_e = downbeats[bar_idx + 1] if bar_idx + 1 < len(downbeats) else pm.get_end_time()
    for n in all_notes:
        if b_s <= n[0] < b_e:
            all_positions.append(get_position_in_bar(n[0], b_s, b_e))

plt.figure(figsize=(10, 3))
plt.bar(range(POSITION_RESOLUTION), np.bincount(all_positions, minlength=POSITION_RESOLUTION))
plt.xlabel("Position within bar (0–15)")
plt.ylabel("Note count")
plt.title("Note onset distribution across bar positions")
plt.xticks(range(POSITION_RESOLUTION))
plt.tight_layout()
plt.show()

## 7. Edge Cases to Watch

Things that go wrong with some Lakh/ADL files:

In [ ]:
# --- 7a. No time signature info ---
# pretty_midi inserts a default 4/4 at t=0 if none is present.
# This means time_signature_changes is never empty — safe to always read it.
print(f"Time sig changes (always >= 1): {len(pm.time_signature_changes)}")
print(f"Default (always present): {pm.time_signature_changes[0].numerator}/{pm.time_signature_changes[0].denominator}")

In [ ]:
# --- 7b. Multiple time signature changes ---
# get_downbeats() handles this automatically.
# For expert descriptions, we need to know which time sig is active per bar.
# The get_time_sig_at() helper above handles this.

unique_sigs = set(f"{ts.numerator}/{ts.denominator}" for ts in pm.time_signature_changes)
print(f"Unique time signatures in this file: {unique_sigs}")

In [ ]:
# --- 7c. Very short or empty bars ---
# Can occur near tempo changes or at the end of a file.
# Guard: skip bars with duration < some threshold, or bar_dur == 0.
n_bars = len(downbeats)
bar_ends = np.append(downbeats[1:], pm.get_end_time())
bar_durs = bar_ends - downbeats
print(f"Bars with duration < 0.1s: {(bar_durs < 0.1).sum()}")
print(f"Empty bars (no notes): {sum(1 for f in bar_features if f['note_count'] == 0)}")

In [ ]:
# --- 7d. Tempo changes ---
# get_downbeats() already accounts for these.
# If we want tempo as an expert feature:
tempo_change_times, tempos = pm.get_tempo_changes()
print(f"Tempo changes: {len(tempos)}")
for t, bpm in zip(tempo_change_times[:5], tempos[:5]):
    print(f"  t={t:.2f}s  {bpm:.1f} BPM")

def get_tempo_at(pm, t):
    """Return BPM at time t."""
    times, tempos = pm.get_tempo_changes()
    if len(tempos) == 0:
        return 120.0
    idx = np.searchsorted(times, t, side='right') - 1
    return tempos[max(0, idx)]

## 8. Vocabulary Design Sketch

What tokens the REMI+ tokenizer and expert description will need.

### REMI+ structural tokens (added to base vocab)
```
<BAR>                    — marks the start of a new bar
<POSITION_0> .. <POSITION_15>  — onset position within bar (16 bins)
<TIME_SIG_4_4>           — active time signature
<TIME_SIG_3_4>
<TIME_SIG_6_8>
... (cover the most common; group rare ones into <TIME_SIG_OTHER>)
```

### Expert description tokens (prepended before each bar's notes)
```
<DESC_START>  <DESC_END>       — delimiters
<DENSITY_0> .. <DENSITY_7>    — 8 bins for notes/sec
<PITCH_LOW_0>..<PITCH_LOW_7>  — 8 bins for lowest pitch in bar
<PITCH_HIGH_0>..<PITCH_HIGH_7>— 8 bins for highest pitch
<VEL_MEAN_0>..<VEL_MEAN_7>   — 8 bins for mean velocity
<POLY_0>..<POLY_3>            — 4 bins for mean polyphony (1, 2, 3, 4+)
<TIME_SIG_4_4> etc.           — shared with structural vocab above
```

Total new tokens: ~1 + 16 + N_time_sigs + 2 + 8+8+8+8+4 = ~60–70 depending on
how many time signatures we cover. Vocab grows from 448 → ~520.

In [ ]:
# --- Survey time signatures across a sample of files ---
# Run this on a few hundred ADL files to decide which TIME_SIG tokens to include

from collections import Counter

def survey_time_signatures(file_list, max_files=200):
    counter = Counter()
    errors = 0
    for path in file_list[:max_files]:
        try:
            p = pretty_midi.PrettyMIDI(str(path))
            for ts in p.time_signature_changes:
                counter[f"{ts.numerator}/{ts.denominator}"] += 1
        except Exception:
            errors += 1
    return counter, errors

# Uncomment and point at your ADL directory:
# adl_files = sorted(Path("../../../data/adl-piano-midi").rglob("*.mid"))
# sig_counts, errs = survey_time_signatures(adl_files)
# print(f"Parse errors: {errs}")
# for sig, count in sig_counts.most_common(15):
#     print(f"  {sig:8s}  {count}")

print("Uncomment above once ADL data is available.")

## 9. Density / Pitch Bin Boundaries

Before writing the expert description vocabulary, we need to choose bin boundaries
for density, pitch, velocity. These should be data-driven from the ADL corpus.
Here we compute them from the current file as a sanity check — repeat over the full
dataset when available.

In [ ]:
densities    = [f["note_density"] for f in bar_features if f["note_count"] > 0]
pitch_mins   = [f["pitch_min"]    for f in bar_features if f["note_count"] > 0]
pitch_maxs   = [f["pitch_max"]    for f in bar_features if f["note_count"] > 0]
vel_means    = [f["velocity_mean"]for f in bar_features if f["note_count"] > 0]
poly_means   = [f["polyphony_mean"] for f in bar_features if f["note_count"] > 0]

N_BINS = 8
for name, values in [("density (notes/s)", densities), ("pitch_min", pitch_mins),
                     ("pitch_max", pitch_maxs), ("vel_mean", vel_means), ("polyphony", poly_means)]:
    arr = np.array(values)
    percentiles = np.percentile(arr, np.linspace(0, 100, N_BINS + 1))
    print(f"{name:22s}  bins: {np.round(percentiles, 1)}")

## Summary

What we've confirmed:
1. `pm.get_downbeats()` reliably gives bar start times — no manual tempo math needed.
2. `pm.time_signature_changes` is never empty (pretty_midi inserts 4/4 default).
3. `np.searchsorted(downbeats, note_times, side='right') - 1` assigns notes to bars correctly.
4. Position within bar = `int((t - bar_start) / bar_dur * RESOLUTION)` with 16 positions is clean.
5. Edge cases (empty bars, short bars, tempo changes) are manageable.

Next steps:
- Run section 8's time signature survey on the full ADL corpus.
- Run section 9's bin boundary computation on the full ADL corpus.
- Write `data_management/tokenizing_remi.py` using these primitives.
- Write `data_management/expert_descriptions.py` for the description prefix tokens.